# 04: LoRA Fine-tune (H3 Stretch Goal)

Trains a LoRA adapter for LLaVA-1.6-7B on the synthetic instruction-image
corpus from §3.3 of the proposal. Designed for **2× T4** with `device_map='auto'`
model sharding — single T4 will OOM with default sequence length.

**Settings:** Accelerator = `GPU T4 x2`, Internet = `On`.

**Datasets to attach:**
- `gvla-weights`
- `gvla-data`  (must include the `synthetic_sample` folder, or your own larger
  synthetic dataset — increase to ~200 examples for meaningful results)

Approximate runtime: ~30–60 min on 2× T4 for 3 epochs over 200 examples.

In [ ]:
import os, subprocess
WEIGHTS = '/kaggle/input/gvla-weights/hf_cache'
DATA    = '/kaggle/input/gvla-data'
os.environ['HF_HOME'] = WEIGHTS
os.environ['TRANSFORMERS_CACHE'] = WEIGHTS
os.environ['TRANSFORMERS_OFFLINE'] = '1'
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv'])

In [ ]:
REPO_URL = 'https://github.com/<your-org>/grounded_vla.git'
!if [ ! -d /kaggle/working/grounded_vla ]; then git clone {REPO_URL} /kaggle/working/grounded_vla; fi
!git -C /kaggle/working/grounded_vla pull
%cd /kaggle/working/grounded_vla
!pip install -q -e .[gpu]

In [ ]:
# Train. The synthetic_sample fixture is tiny (3 examples) and only useful as a
# pipeline check; for real H3 numbers, swap to a 200-example synthetic.jsonl.
from grounded_vla.lora import train_lora, LoRAConfig
from pathlib import Path

JSONL = f'{DATA}/synthetic_sample/synthetic_sample.jsonl'  # or your real synthetic.jsonl
IMGS  = f'{DATA}/synthetic_sample'
OUT   = Path('/kaggle/working/checkpoints/llava-lora-r1')
OUT.mkdir(parents=True, exist_ok=True)

cfg = LoRAConfig(
    base_model=f'{WEIGHTS}/llava-v1.6-mistral-7b-hf',
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    learning_rate=2e-4,
    per_device_batch_size=1,
    gradient_accumulation_steps=8,
    num_epochs=3,
    max_seq_len=768,  # smaller than default to stay safe on T4
)
train_lora(jsonl_path=JSONL, images_dir=IMGS, output_dir=OUT, config=cfg)

In [ ]:
# Verify adapter was saved.
import os
for f in sorted(os.listdir('/kaggle/working/checkpoints/llava-lora-r1')):
    print(f)

## Next step

**Save Version** → name the output dataset `gvla-lora-r1`. Notebook 05 mounts it
to evaluate the H3 ablation.